# 服飾商品資料庫 (Clothing Database) - Google Colab 測試與串接程式
本筆記本用於測試與驗證 **TiDB Cloud (MySQL 相容) 資料庫** 連線、資料表結構、`image_url` 商品圖片對應，並執行查詢題目（A1～C2）。

### 作業評分項目涵蓋：
1. **MySQL / TiDB Cloud 雲端資料庫串接驗證**
2. **Colab 程式連線與資料表測試**
3. **60 項服飾商品圖片 URL (image_url) 完整性確認**
4. **SQL 核心查詢與進階統計分析**

## 步驟 0：安裝必要 Python 套件

In [ ]:
!pip install -q pymysql cryptography pandas matplotlib seaborn

## 步驟 1：設定 TiDB Cloud 連線資訊
請填入您在 TiDB Cloud 建立的連線資訊（Host、User、Password、Database）。

In [ ]:
import os
import ssl
import pymysql
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 請在此輸入您 TiDB Cloud 的連線帳密 (或透過環境變數傳入)
TIDB_CONFIG = {
    "host": os.getenv("TIDB_HOST", "gateway01.ap-northeast-1.prod.aws.tidbcloud.com"),
    "port": int(os.getenv("TIDB_PORT", "4000")),
    "user": os.getenv("TIDB_USER", "root"),
    "password": os.getenv("TIDB_PASSWORD", "your_password"),
    "database": os.getenv("TIDB_DATABASE", "clothing_store"),
    "charset": "utf8mb4",
    "ssl": ssl.create_default_context(),
    "cursorclass": pymysql.cursors.DictCursor
}

def get_db_connection():
    return pymysql.connect(**TIDB_CONFIG)

print("連線設定完成！")

## 步驟 2：測試資料庫連線並檢視所有資料表

In [ ]:
try:
    conn = get_db_connection()
    with conn.cursor() as cursor:
        cursor.execute("SHOW TABLES;")
        tables = cursor.fetchall()
        print("✅ 成功連線至 TiDB Cloud！資料表清單：")
        for t in tables:
            print(f" - {list(t.values())[0]}")
    conn.close()
except Exception as e:
    print("❌ 連線失敗，請檢查 TIDB_CONFIG 設定：", e)

## 步驟 3：驗證 60 件服飾與 image_url 圖片網址對應
確認 `clothing` 資料表中的 `image_url` 欄位已完整建立，且能對應至 GitHub Pages 靜態圖床。

In [ ]:
query = """
    SELECT c.product_id, c.product_name, ca.category_name, b.brand_name,
           c.color, c.size, c.price, c.stock_qty, c.image_url
    FROM clothing c
    JOIN categories ca ON c.category_id = ca.category_id
    JOIN brands b ON c.brand_id = b.brand_id
    ORDER BY c.product_id ASC;
"""

try:
    conn = get_db_connection()
    df_clothing = pd.read_sql(query, conn)
    conn.close()
    print(f"✅ 成功讀取 {len(df_clothing)} 件服飾商品！")
    display(df_clothing.head(10))
except Exception as e:
    print("讀取失敗：", e)

## 步驟 4：執行作業題目測試查詢 (A1～A5, B1～B3)

In [ ]:
queries = {
    "A1: 所有上衣商品名稱與價格 (依價格降冪)": """
        SELECT c.product_name, c.price
        FROM clothing c
        JOIN categories cat ON c.category_id = cat.category_id
        WHERE cat.category_name = '上衣'
        ORDER BY c.price DESC;
    """,
    "A2: Urban Leaf 品牌價格大於 1,000 元的商品": """
        SELECT c.product_id, c.product_name, c.price
        FROM clothing c
        JOIN brands b ON c.brand_id = b.brand_id
        WHERE b.brand_name = 'Urban Leaf' AND c.price > 1000;
    """,
    "A5: 名稱包含「外套」的商品": """
        SELECT c.product_name, b.brand_name, c.price
        FROM clothing c
        JOIN brands b ON c.brand_id = b.brand_id
        WHERE c.product_name LIKE '%外套%';
    """,
    "B1: 商品款數超過 2 款的分類與平均價格": """
        SELECT cat.category_name, COUNT(c.product_id) AS product_count, AVG(c.price) AS avg_price
        FROM categories cat
        JOIN clothing c ON cat.category_id = c.category_id
        GROUP BY cat.category_id, cat.category_name
        HAVING COUNT(c.product_id) > 2;
    """,
    "B2: 定價最高商品 (子查詢 MAX)": """
        SELECT product_name, price
        FROM clothing
        WHERE price = (SELECT MAX(price) FROM clothing);
    """
}

try:
    conn = get_db_connection()
    for title, q in queries.items():
        print(f"\n=== {title} ===")
        res = pd.read_sql(q, conn)
        display(res)
    conn.close()
except Exception as e:
    print("查詢執行異常：", e)

## 步驟 5：視覺化分析 — 服飾分類與價格分布

In [ ]:
try:
    conn = get_db_connection()
    df_stats = pd.read_sql("""
        SELECT ca.category_name, COUNT(*) AS count, AVG(c.price) AS avg_price
        FROM clothing c
        JOIN categories ca ON c.category_id = ca.category_id
        GROUP BY ca.category_name;
    """, conn)
    conn.close()

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.bar(df_stats['category_name'], df_stats['count'], color='#e8788b')
    plt.title('各分類商品數量')
    plt.xticks(rotation=45)

    plt.subplot(1, 2, 2)
    plt.bar(df_stats['category_name'], df_stats['avg_price'], color='#b44c63')
    plt.title('各分類平均價格 (NT$)')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()
except Exception as e:
    print("繪圖失敗：", e)